# Hash Sanity Check — Is `random_negative` noise visible to LSR?

Runs **1 round, α=0.0, mechanism mode** and checks whether the aggregated model hash
differs from the old `5706c0d7...` baseline (exp_02/03/04a).

- **PASS** (hash differs) → random_negative noise is affecting training; proceed to full exp_04b.
- **FAIL** (hash matches) → noise is still invisible; deeper problem, reframe contribution.

In [1]:
from pathlib import Path

REPO_URL    = "https://github.com/Abishek-Chakravarthy/fed-rag.git"
REPO_BRANCH = "q-fedrag2"
BASELINE_HASH = "5706c0d7"  # R1 hash from exp_02, exp_03, exp_04a

WORKSPACE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
REPO_DIR  = WORKSPACE / "fed-rag"
EXP_DIR   = REPO_DIR / "zz_coderuns" / "quality_aware_fedrag" / "mechanism_benchmark"

print(f"WORKSPACE : {WORKSPACE}")
print(f"BASELINE  : {BASELINE_HASH}")

WORKSPACE : /kaggle/working
BASELINE  : 5706c0d7


In [2]:
import os, shutil, subprocess, sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
                "--single-branch", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "protobuf>=4.25.3,<6", "accelerate", "datasets<3.0.0", "flwr==1.22.0",
    "pyarrow", "pydantic", "pydantic-settings", "transformers==4.48.0",
    "sentence-transformers==3.4.1", "peft", "pandas", "tqdm"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-e", str(REPO_DIR), "--no-deps"], check=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
print("Clone + install complete")

Cloning into '/kaggle/working/fed-rag'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.25 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.6.1 which is incompatible.
black 26.3.1 requires pathspec>=1.0.0, but you have pathspec 0.12.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 44.0.3 which is incompati

Clone + install complete


In [3]:
# Run 1 round, alpha=0.0 — just enough to get the R1 model hash
cmd = [
    sys.executable, "-u", "federated_noisy_qa.py",
    "--alpha", "0.0",
    "--seed", "42",
    "--benchmark-mode", "mechanism",
    "--noise-mode", "random_negative",
    "--noise-ratio", "0.0",
    "--rounds", "1",
    "--local-epochs", "1",
]

print("Running:", " ".join(cmd))
process = subprocess.Popen(
    cmd, cwd=EXP_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end="")
process.wait()
if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, cmd)

Running: /usr/bin/python3 -u federated_noisy_qa.py --alpha 0.0 --seed 42 --benchmark-mode mechanism --noise-mode random_negative --noise-ratio 0.0 --rounds 1 --local-epochs 1
2026-04-25 13:06:52.781443: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777122413.179911      81 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777122413.294802      81 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777122414.253765      81 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777122414.253828      81 computation_placer.cc:177] computation placer alr

In [4]:
import pandas as pd

csv_path = EXP_DIR / "output_csv_files" / \
    "results_alpha_0.0_track_mechanism_seed_42_mode_random_negative_ratio_0p00_r1_e1.csv"

df = pd.read_csv(csv_path)
r1_hash = df.loc[0, "aggregated_model_hash"]

print(f"R1 model hash  : {r1_hash}")
print(f"Baseline hash  : {BASELINE_HASH}")
print()

if r1_hash.startswith(BASELINE_HASH):
    print("❌ FAIL — hash MATCHES baseline. random_negative noise is still invisible to LSR.")
    print("   Do NOT proceed to full exp_04b. Reframe contribution.")
else:
    print("✅ PASS — hash DIFFERS from baseline. random_negative noise IS affecting training.")
    print("   Proceed to full exp_04b (4 rounds × 4 alphas).")

display(df[["round", "aggregated_model_hash",
            "server_val_mrr", "server_val_ndcg_at_k"]])

R1 model hash  : 5706c0d741234daef003fb82324c5b07
Baseline hash  : 5706c0d7

❌ FAIL — hash MATCHES baseline. random_negative noise is still invisible to LSR.
   Do NOT proceed to full exp_04b. Reframe contribution.


,round,aggregated_model_hash,server_val_mrr,server_val_ndcg_at_k
0,1,5706c0d741234daef003fb82324c5b07,0.02485,0.03466
